In [4]:
"""
Évaluation des modèles OpenRouter sur le classeur SHORT-ANSWER (réponse libre courte)
--------------------------------------------------------------------------------
Colonnes attendues dans la feuille :
  item_id, Axe, Difficulty, intent_group, intent_category, input,
  answer, primary_accepted_variants, second_accepted_variants, all_supporting_sources

Le script ajoute pour chaque modèle :
  response_<nom_court>_raw    -> texte brut renvoyé par le modèle (tel quel)
  response_<nom_court>_parse  -> méthode de parsing utilisée : "tag" (balise trouvée,
                                  cas normal) ou "fallback" (le modèle n'a pas respecté
                                  le format demandé, on garde tout le texte nettoyé)
  response_<nom_court>        -> réponse finale extraite (ou "EMPTY" si vide)

CE FICHIER NE FAIT PLUS LE SCORING. Il ne fait que collecter les réponses des
modèles. Le scoring "propre" (exact / inclusion / flou, ventilé par Axe/
Difficulty/intent_group/intent_category, avec classeur Excel dédié) est fait
par le script séparé sa_scoring.py, à partir du fichier produit ici.
Un print_scores() très basique reste en fin de run, juste comme indicateur
rapide pendant le développement — les chiffres qui font foi sont ceux du
classeur produit par sa_scoring.py.

--- Format de sortie imposé au modèle (important pour un parsing fiable) ---
Le prompt demande au modèle d'encadrer sa réponse finale entre deux balises
littérales <FINAL_ANSWER> ... </FINAL_ANSWER>. Le modèle peut réfléchir ou
expliquer AVANT la balise ouvrante (utile pour les modèles à raisonnement
obligatoire), mais rien n'est censé suivre la balise fermante. On extrait le
contenu entre balises avec une regex ; si aucune balise n'est trouvée (modèle
qui n'a pas respecté la consigne), on retombe sur tout le texte nettoyé et on
marque la ligne comme "fallback" dans response_<modele>_parse, pour pouvoir
repérer facilement les modèles peu fiables sur le format.

Caractéristiques (identiques au script MCQ d'origine) :
- Appels concurrents (MAX_WORKERS, 10 par défaut).
- Gestion automatique du champ "reasoning" : on tente d'abord désactivé ; si
  l'endpoint répond « Reasoning is mandatory... », le modèle bascule en reasoning
  activé (puis, en dernier recours, le champ est omis). Le mode qui marche est
  mémorisé pour tous les appels suivants de ce modèle.
- Sauvegarde régulière + reprise : les lignes déjà remplies (response_<modele>
  non vide) sont sautées, donc il suffit de ré-exécuter la cellule après un
  crash ou un Ctrl-C : seules les lignes manquantes sont relancées, jamais les
  lignes déjà répondues.
- Aucune erreur réseau/API/parsing n'arrête le run.

À faire avant de lancer :
1. Mets ta clé dans OPENROUTER_API_KEY (variable d'environnement ou en dur).
2. Vérifie EXCEL_PATH et SHEET_NAME.
"""

import os
import re
import time
import random
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests

# ============================== CONFIG ==============================

EXCEL_PATH = "short_answers_final.xlsx"
SHEET_NAME = "Feuil1"

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "sk-or-v1dd4316815aedd6a29707afd")
API_URL = "https://openrouter.ai/api/v1/chat/completions"

# nom complet OpenRouter -> nom court utilisé dans les noms de colonnes
MODEL_MAP = {
    "google/gemma-4-26b-a4b-it": "gemma4_26b",
    "qwen/qwen3.6-27b": "qwen3_6_27b",
    "qwen/qwen3.5-9b": "qwen3_5_9b",
    "cohere/command-r-08-2024": "commandr",
    "google/gemma-4-31b-it": "gemma4_31b",
    "openai/gpt-oss-20b": "gptoss_20b",
    "openai/gpt-5": "gpt5",
    "anthropic/claude-sonnet-5": "claude_sonnet5",
    "google/gemini-3.6-flash": "gemini3_6_flash",
    "x-ai/grok-4.20": "grok4_20",
    "mistralai/mistral-small-3.2-24b-instruct": "mistral_24b",
    "mistralai/mistral-saba": "mistral_saba",
}

# --- structure du short-answer (utilisée seulement pour le print_scores indicatif) ---
VARIANT_COLS = ["answer", "primary_accepted_variants", "second_accepted_variants"]
VARIANT_SEP = "|"

# --- concurrence / robustesse ---
MAX_WORKERS = 10           # nb d'appels API en parallèle
MAX_RETRIES = 4            # nb de tentatives par appel en cas d'erreur transitoire
RETRY_BACKOFF_SECONDS = 5  # backoff de base (multiplié par le n° de tentative)
REQUEST_TIMEOUT = 90       # timeout HTTP en secondes
SAVE_EVERY = 25            # sauvegarder le classeur toutes les N lignes terminées
JITTER_BEFORE_CALL = 0.15  # petit décalage aléatoire avant chaque appel

# ============================== PROMPT ==============================

ANSWER_OPEN_TAG = "<FINAL_ANSWER>"
ANSWER_CLOSE_TAG = "</FINAL_ANSWER>"

def build_prompt(row) -> str:
    """Construit le prompt pour une question à réponse courte, avec un format
    de sortie strict (balises) pour rendre le parsing fiable."""
    prompt = (
        f"{row['input']}\n\n"
        "جاوب على السؤال بالدارجة المغربية ولا بالعربية الفصحى.\n"
        "خاصك تكتب الجواب النهائي ديالك (كلمة ولا جملة قصيرة، بلا شرح) بين هاد "
        "العلامتين بالضبط، مرة وحدة فقط، بلا ما تبدل شكلهم:\n"
        f"{ANSWER_OPEN_TAG}جوابك هنا{ANSWER_CLOSE_TAG}\n\n"
        "يمكن ليك تفكر ولا تشرح شوية قبل هاد العلامتين إلا كان ضروري بالنسبة "
        "ليك، ولكن الجواب النهائي والمختصر خاصو يكون بين العلامتين، وماكاينش "
        f"تكتب حتى حرف بعد {ANSWER_CLOSE_TAG}."
    )
    return prompt

# ============================== PARSING ==============================

ANSWER_TAG_PAT = re.compile(
    re.escape(ANSWER_OPEN_TAG) + r'(.*?)' + re.escape(ANSWER_CLOSE_TAG),
    re.DOTALL | re.IGNORECASE,
)

def clean_answer(raw_text) -> str:
    """Nettoyage léger (espaces, guillemets superflus) sans dénaturer le contenu."""
    if raw_text is None:
        return ""
    text = str(raw_text).strip()
    text = text.strip(" \n\t\"'“”«»")
    return text

def extract_final_answer(raw_text):
    """
    Retourne (reponse, methode) avec methode in {"tag", "fallback", "empty"} :
      - "tag"      : balises <FINAL_ANSWER>...</FINAL_ANSWER> trouvées et non vides
                     (cas nominal).
      - "fallback" : pas de balises exploitables -> on garde tout le texte nettoyé.
      - "empty"    : rien d'exploitable du tout.
    S'il y a plusieurs occurrences de la balise, on prend la dernière (au cas où
    le modèle recopie l'exemple du prompt avant sa vraie réponse).
    """
    if not raw_text:
        return "", "empty"
    text = str(raw_text)
    matches = ANSWER_TAG_PAT.findall(text)
    if matches:
        candidate = clean_answer(matches[-1])
        if candidate:
            return candidate, "tag"
    cleaned = clean_answer(text)
    if cleaned:
        return cleaned, "fallback"
    return "", "empty"

# ============================== APPEL API ==============================

# mode "reasoning" par modèle : "off" -> désactivé, "on" -> activé, "omit" -> champ absent
REASONING_MODE = {}
_reasoning_lock = threading.Lock()
_thread_local = threading.local()

REASONING_MANDATORY_PAT = re.compile(r"reasoning is mandatory", re.IGNORECASE)

def get_session() -> requests.Session:
    """Une session HTTP par thread (connexions réutilisées, thread-safe)."""
    s = getattr(_thread_local, "session", None)
    if s is None:
        s = requests.Session()
        adapter = requests.adapters.HTTPAdapter(pool_connections=MAX_WORKERS,
                                                pool_maxsize=MAX_WORKERS)
        s.mount("https://", adapter)
        _thread_local.session = s
    return s

def get_mode(model: str) -> str:
    with _reasoning_lock:
        return REASONING_MODE.setdefault(model, "off")

def set_mode(model: str, mode: str):
    with _reasoning_lock:
        if REASONING_MODE.get(model) != mode:
            REASONING_MODE[model] = mode
            print(f"  >>> [{model}] bascule du paramètre reasoning en '{mode}'")

def build_payload(model: str, prompt: str, mode: str) -> dict:
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
    }
    if mode == "off":
        payload["reasoning"] = {"enabled": False}
    elif mode == "on":
        payload["reasoning"] = {"enabled": True}
    # mode == "omit" -> on n'envoie rien
    return payload

def call_openrouter(model: str, prompt: str):
    """
    Appelle OpenRouter avec retries + négociation automatique du champ reasoning.
    Retourne (contenu_texte, erreur) : un seul des deux est non-None.
    """
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    session = get_session()
    last_error = None
    tried_modes = set()

    for attempt in range(1, MAX_RETRIES + 1):
        mode = get_mode(model)
        tried_modes.add(mode)
        try:
            resp = session.post(API_URL, headers=headers,
                                json=build_payload(model, prompt, mode),
                                timeout=REQUEST_TIMEOUT)
        except requests.exceptions.RequestException as e:
            last_error = f"request_exception: {e}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt + random.uniform(0, 1))
            continue

        if resp.status_code == 200:
            try:
                data = resp.json()
                msg = data["choices"][0]["message"]
                content = msg.get("content")
                if not content:  # certains modèles renvoient tout dans "reasoning"
                    content = msg.get("reasoning") or ""
                return content, None
            except Exception as e:
                return None, f"unexpected_response_format: {e} | raw={resp.text[:300]}"

        body = resp.text[:500]

        # --- reasoning obligatoire, ou champ reasoning problématique ---
        if resp.status_code in (400, 404, 422) and REASONING_MANDATORY_PAT.search(body):
            set_mode(model, "on")
            continue  # on retente tout de suite, sans backoff
        if resp.status_code in (400, 404, 422) and "reasoning" in body.lower():
            if "omit" not in tried_modes:
                set_mode(model, "omit")
                continue

        if resp.status_code == 429:
            last_error = f"rate_limited (429): {body[:200]}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt * 2 + random.uniform(0, 2))
        elif 500 <= resp.status_code < 600:
            last_error = f"server_error ({resp.status_code}): {body[:200]}"
            time.sleep(RETRY_BACKOFF_SECONDS * attempt + random.uniform(0, 1))
        else:
            return None, f"http_{resp.status_code}: {body[:300]}"

    return None, last_error

# ============================== I/O EXCEL ==============================

def load_all_sheets(path: str) -> dict:
    if not Path(path).exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")
    return pd.read_excel(path, sheet_name=None)

def save_all_sheets(sheets: dict, path: str):
    """Écrit dans un fichier temporaire puis remplace l'original (évite un fichier
    corrompu si le process est tué pendant l'écriture)."""
    p = Path(path)
    tmp_path = p.with_name(p.stem + ".tmp" + p.suffix)
    with pd.ExcelWriter(tmp_path, engine="openpyxl") as writer:
        for name, sheet_df in sheets.items():
            sheet_df.to_excel(writer, sheet_name=name, index=False)
    tmp_path.replace(path)

# ============================== BOUCLE PRINCIPALE ==============================

# Variables globales : si une sauvegarde échoue et que le run est interrompu,
# "sheets" et "df" restent accessibles depuis le notebook -> sauvegarde manuelle :
#   save_all_sheets(sheets, EXCEL_PATH)
sheets = None
df = None

def _work(model_full, idx, prompt):
    """Exécuté dans un thread : ne touche jamais au dataframe."""
    if JITTER_BEFORE_CALL:
        time.sleep(random.uniform(0, JITTER_BEFORE_CALL))
    raw_text, error = call_openrouter(model_full, prompt)
    if error is not None:
        return idx, raw_text, error, None, None
    answer, method = extract_final_answer(raw_text)
    return idx, raw_text, error, answer, method

def run_evaluation():
    global sheets, df
    sheets = load_all_sheets(EXCEL_PATH)
    df = sheets[SHEET_NAME]
    if "input" not in df.columns:
        raise ValueError("Colonne 'input' introuvable dans la feuille.")

    for model_full, model_short in MODEL_MAP.items():
        col_raw = f"response_{model_short}_raw"
        col_final = f"response_{model_short}"
        col_parse = f"response_{model_short}_parse"

        for c in (col_raw, col_final, col_parse):
            if c not in df.columns:
                df[c] = pd.NA

        todo = [i for i in df.index if pd.isna(df.at[i, col_final])]
        print(f"\n=== {model_full} ({model_short}) — {len(todo)} ligne(s) restante(s) ===")
        if not todo:
            continue

        done = 0
        done_since_save = 0
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futures = {}
            for idx in todo:
                try:
                    prompt = build_prompt(df.loc[idx])
                except Exception as e:
                    df.at[idx, col_raw] = f"[PROMPT_ERROR] {e}"
                    print(f"  [{idx}] impossible de construire le prompt : {e}")
                    continue
                futures[pool.submit(_work, model_full, idx, prompt)] = idx

            for fut in as_completed(futures):
                idx = futures[fut]
                item_id = df.at[idx, "item_id"] if "item_id" in df.columns else idx
                try:
                    idx, raw_text, error, answer, method = fut.result()
                    if error is not None:
                        df.at[idx, col_raw] = f"[ERROR] {error}"
                        # col_final reste NA -> sera retenté au prochain run
                        print(f"  [{item_id}] ERREUR API: {error}")
                    else:
                        df.at[idx, col_raw] = raw_text
                        df.at[idx, col_final] = answer if answer else "EMPTY"
                        df.at[idx, col_parse] = method
                        if method == "fallback":
                            print(f"  [{item_id}] balises absentes, fallback texte brut: "
                                  f"{str(raw_text)[:120]!r}")
                except Exception as e:
                    df.at[idx, col_raw] = f"[UNEXPECTED_ERROR] {e}"
                    print(f"  [{item_id}] ERREUR INATTENDUE: {e}")

                done += 1
                done_since_save += 1
                if done % 50 == 0:
                    print(f"  ... {done}/{len(futures)}")
                if done_since_save >= SAVE_EVERY:
                    try:
                        sheets[SHEET_NAME] = df
                        save_all_sheets(sheets, EXCEL_PATH)
                    except Exception as e:
                        print(f"  (sauvegarde échouée, on continue quand même: {e})")
                    done_since_save = 0

        # sauvegarde de fin de modèle
        try:
            sheets[SHEET_NAME] = df
            save_all_sheets(sheets, EXCEL_PATH)
        except Exception as e:
            print(f"Sauvegarde finale échouée pour {model_short}: {e}")

    print("\nTerminé.")
    print_scores()

# ============================== SCORE INDICATIF (rapide) ==============================
# Juste un aperçu grossier pendant le développement. Le scoring définitif (exact /
# inclusion / flou, ventilé par dimension, classeur dédié) est fait par sa_scoring.py.

def _normalize_quick(s) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    s = re.sub(r'[إأآا]', 'ا', s)
    s = re.sub(r'[ىي]', 'ي', s)
    s = re.sub(r'ة', 'ه', s)
    s = re.sub(r'[^\w\s]', '', s, flags=re.UNICODE)
    s = re.sub(r'\s+', ' ', s).strip()
    return s.lower()

def _quick_accepted_variants(row) -> list:
    variants = []
    for col in VARIANT_COLS:
        if col not in row.index or pd.isna(row[col]):
            continue
        for part in str(row[col]).split(VARIANT_SEP):
            part = part.strip()
            if part:
                variants.append(part)
    return variants

def print_scores():
    if df is None:
        return
    print("\n--- Score indicatif rapide (voir sa_scoring.py pour les vrais chiffres) ---")
    for model_short in MODEL_MAP.values():
        col = f"response_{model_short}"
        if col not in df.columns:
            continue
        n_missing = int(df[col].isna().sum())
        n_empty = int((df[col].astype(str).str.strip() == "EMPTY").sum())
        answered, correct = 0, 0
        for idx in df.index:
            pred = df.at[idx, col]
            if pd.isna(pred) or str(pred).strip() in ("", "EMPTY"):
                continue
            answered += 1
            pred_norm = _normalize_quick(pred)
            for v in _quick_accepted_variants(df.loc[idx]):
                v_norm = _normalize_quick(v)
                if v_norm and (pred_norm == v_norm or v_norm in pred_norm or pred_norm in v_norm):
                    correct += 1
                    break
        if answered == 0:
            print(f"{model_short:>18} : aucune réponse exploitable "
                  f"(vides={n_empty}, manquants={n_missing})")
            continue
        acc = correct / answered
        print(f"{model_short:>18} : {acc:.1%} sur {answered} items "
              f"(vides={n_empty}, manquants={n_missing})")


try:
    run_evaluation()
except KeyboardInterrupt:
    print("\nInterrompu manuellement — les résultats déjà obtenus sont sauvegardés. "
          "Ré-exécute la cellule pour reprendre là où tu t'es arrêté.")



=== google/gemma-4-26b-a4b-it (gemma4_26b) — 0 ligne(s) restante(s) | dépense cumulée jusqu'ici: $0.0000 ===

=== qwen/qwen3.6-27b (qwen3_6_27b) — 0 ligne(s) restante(s) | dépense cumulée jusqu'ici: $0.0000 ===

=== qwen/qwen3.5-9b (qwen3_5_9b) — 0 ligne(s) restante(s) | dépense cumulée jusqu'ici: $0.0000 ===

=== cohere/command-r-08-2024 (commandr) — 0 ligne(s) restante(s) | dépense cumulée jusqu'ici: $0.0000 ===

=== google/gemma-4-31b-it (gemma4_31b) — 0 ligne(s) restante(s) | dépense cumulée jusqu'ici: $0.0000 ===

=== openai/gpt-oss-20b (gptoss_20b) — 0 ligne(s) restante(s) | dépense cumulée jusqu'ici: $0.0000 ===

=== openai/gpt-5 (gpt5) — 0 ligne(s) restante(s) | dépense cumulée jusqu'ici: $0.0000 ===

=== anthropic/claude-sonnet-5 (claude_sonnet5) — 109 ligne(s) restante(s) | dépense cumulée jusqu'ici: $0.0000 ===
  (sauvegarde échouée, on continue quand même: [WinError 5] Accès refusé: 'short_answers_final.tmp.xlsx' -> 'short_answers_final.xlsx')
  [SA_066] balises absentes, 

In [6]:
5

5

In [ ]:
5

In [1]:
5

5